In [1]:
import pandas as pd
from datetime import date

In [2]:
encounters = pd.read_csv('C:/Users/Esteban Wong/analyst-take-home-task/datasets/encounters.csv',usecols=['Id','START','STOP','PATIENT','REASONDESCRIPTION'])
patients = pd.read_csv('C:/Users/Esteban Wong/analyst-take-home-task/datasets/patients.csv',usecols=['Id','BIRTHDATE','DEATHDATE'])
medications = pd.read_csv('C:/Users/Esteban Wong/analyst-take-home-task/datasets/medications.csv',usecols=['PATIENT','DESCRIPTION','REASONDESCRIPTION'])

encounters = encounters.rename(columns={'Id':'ENCOUNTER_ID','START':'HOSPITAL_ENCOUNTER_DATE','PATIENT': 'PATIENT_ID'})
patients = patients.rename(columns={'Id':'PATIENT_ID'})
medications = medications.rename(columns={'PATIENT':'PATIENT_ID'})

# Part 1: Assemble the project cohort

### The patient’s visit is an encounter for drug overdose

In [3]:
overdose = encounters[encounters['REASONDESCRIPTION'].str.contains('Drug overdose',na = False)]

### The hospital encounter occurs after July 15, 1999

In [4]:
overdose['HOSPITAL_ENCOUNTER_DATE'] = pd.to_datetime(overdose['HOSPITAL_ENCOUNTER_DATE'])
overdose_after_071599 = overdose[overdose['HOSPITAL_ENCOUNTER_DATE'] > pd.to_datetime('1999-07-15')]

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


### The patient’s age at time of encounter is between 18 and 35 (Patient is considered to be 35 until turning 36)

In [5]:
# NOTE: medications to patients table could not be joined on ENCOUNTER_ID, PATIENT_ID was used instead
# this later causes the death to be shown accross the patient's record instead of at a given encounter

overdose_with_bd = pd.merge(overdose_after_071599, patients, on = 'PATIENT_ID', how='left')
overdose_with_bd['BIRTHDATE'] = pd.to_datetime(overdose_with_bd['BIRTHDATE'])
overdose_with_bd['AGE_AT_VISIT'] = (overdose_with_bd['HOSPITAL_ENCOUNTER_DATE'] - overdose_with_bd['BIRTHDATE']).dt.days / 365.25
all_criteria = overdose_with_bd[(overdose_with_bd['AGE_AT_VISIT'] >= 18) & (overdose_with_bd['AGE_AT_VISIT'] <= 35)]

# Part 2: Create additional fields

##### DEATH_AT_VISIT_IND - Indicator if the patient died during the drug overdose encounter. Leave N/A if patient has not died. {NaN/1}

In [6]:
all_criteria.loc[all_criteria['DEATHDATE'].notna(), 'DEATH_AT_VISIT_IND'] = 1

C:\ProgramData\Anaconda3\lib\site-packages\pandas\core\indexing.py:362: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  self.obj[key] = _infer_fill_value(value)
C:\ProgramData\Anaconda3\lib\site-packages\pandas\core\indexing.py:543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  self.obj[item] = s


#### COUNT_CURRENT_MEDS - Count of active medications at the start of the drug overdose encounter {Num}

In [7]:
# NOTE: medications to patients table could not be joined on ENCOUNTER_ID, PATIENT_ID was used instead
# meds were therefore shown for the patient instead of speciic encounters
overdosed_medications = pd.merge(all_criteria[['ENCOUNTER_ID','PATIENT_ID']], medications, on='PATIENT_ID', how='left')
overdosed_medications['COUNT_CURRENT_MEDS'] = overdosed_medications.groupby('ENCOUNTER_ID')['ENCOUNTER_ID'].transform('count')

needed_overdosed_medications = overdosed_medications[['PATIENT_ID','COUNT_CURRENT_MEDS']]
all_criteria = pd.merge(all_criteria,needed_overdosed_medications.drop_duplicates(subset=['PATIENT_ID', 'COUNT_CURRENT_MEDS']), on='PATIENT_ID', how='left')

#### CURRENT_OPIOID_IND - if the patient had at least one active medication at the start of the overdose encounter that is on the Opioids List (provided below)	{0/1}

In [8]:
search_strings = ['Hydromorphone 325 MG', 'Fentanyl 100 MCG','Oxycodone-acetaminophen 100ML','Amlodipine 5 MG / Fentanyl 100 MCG / Olmesartan medoxomil 20 MG']
overdosed_medications['CURRENT_OPIOID_IND'] = overdosed_medications['DESCRIPTION'].isin(search_strings)


prescribed_opioid = overdosed_medications[overdosed_medications['CURRENT_OPIOID_IND']==True]
needed_prescribed_opioid = prescribed_opioid[['PATIENT_ID','CURRENT_OPIOID_IND']]

# NOTE: medications to patients table could not be joined on ENCOUNTER_ID, PATIENT_ID was used instead
# opioid was therefore shown for the patient instead of speciic encounters
all_criteria = pd.merge(all_criteria,needed_prescribed_opioid.drop_duplicates(subset=['PATIENT_ID', 'CURRENT_OPIOID_IND']), on='PATIENT_ID', how='left')
all_criteria['CURRENT_OPIOID_IND'] = all_criteria['CURRENT_OPIOID_IND'].replace({True: 1}).fillna(0)

#### READMISSION_90_DAY_IND - Indicator if the visit resulted in a subsequent readmission within 90 days {0/1}

In [9]:
all_criteria = all_criteria.sort_values(by=['PATIENT_ID', 'HOSPITAL_ENCOUNTER_DATE']).reset_index(drop=True)
all_criteria['NEXT_START_ADMISSION'] = pd.to_datetime(all_criteria.groupby('PATIENT_ID')['HOSPITAL_ENCOUNTER_DATE'].shift(-1))
all_criteria['STOP'] = pd.to_datetime(all_criteria['STOP'])
all_criteria['DAYS_TO_NEXT_ADMISSION'] = (all_criteria['NEXT_START_ADMISSION'] - all_criteria['STOP']).dt.days
all_criteria['READMISSION_90_DAY_IND'] = (all_criteria['DAYS_TO_NEXT_ADMISSION'] <= 90).astype(int)

#### READMISSION_30_DAY_IND - Indicator if the visit resulted in a subsequent readmission within 30 days {0/1}

In [10]:
all_criteria['READMISSION_30_DAY_IND'] = (all_criteria['DAYS_TO_NEXT_ADMISSION'] <= 30).astype(int)

#### FIRST_READMISSION_DATE - Date of the first readmission for drug overdose within 90 days. Leave N/A if no readmissions for drug overdose within 90 days.	{Date/time}

In [11]:
first_readmission_dates = all_criteria[all_criteria['READMISSION_90_DAY_IND']==1].groupby('PATIENT_ID')['NEXT_START_ADMISSION'].transform('min')
all_criteria['FIRST_READMISSION_DATE'] = first_readmission_dates
all_criteria['FIRST_READMISSION_DATE'] = all_criteria.groupby('PATIENT_ID')['FIRST_READMISSION_DATE'].ffill()

# Email questions answers based on criteria (completed above)

### Total Patient Encounters (Visits)

In [12]:
all_criteria['ENCOUNTER_ID'].nunique()

401

### Distinct Patients

In [13]:
all_criteria['PATIENT_ID'].nunique()

191

### Encounters where patient was on active Opioid Rx

In [14]:
len(all_criteria[all_criteria['CURRENT_OPIOID_IND']==1])

105

### Deaths during drug overdose encounter

In [15]:
all_criteria[all_criteria['DEATH_AT_VISIT_IND'].notna()]['PATIENT_ID'].nunique()

37

### Readmissions within 90 days

In [16]:
len(all_criteria[all_criteria['READMISSION_90_DAY_IND']==1])

16

### Readmissions within 30 days

In [17]:
len(all_criteria[all_criteria['READMISSION_30_DAY_IND']==1])

4

In [18]:
final_file = all_criteria[['PATIENT_ID','ENCOUNTER_ID','HOSPITAL_ENCOUNTER_DATE','AGE_AT_VISIT','DEATH_AT_VISIT_IND','COUNT_CURRENT_MEDS','CURRENT_OPIOID_IND','READMISSION_90_DAY_IND','READMISSION_30_DAY_IND','FIRST_READMISSION_DATE']]
final_file.to_csv('C:/Users/Esteban Wong/analyst-take-home-task/datasets/Esteban_Wong.csv')